# **초급 프로젝트 3팀 - Pill-터링**

## **프로젝트 보고서**
##### 작성자: 송우현

### **1. 역할 및 목표 (Role & Objective)**

**역할**: Experimentation Lead

**주요 임무**: 다양한 모델 실험 주도, 하이퍼파라미터 튜닝, 모델 성능 평가 및 최적화 전략 수립.

**핵심 가치**: 프로젝트 초기부터 데이터의 특성을 분석하고, 실험을 통해 팀의 모델 성능을 극한으로 끌어올리는 것을 목표로 함.

### **2. 주요 활동 및 실험 과정 (Activities & Experiments)**

**Phase 1: 데이터 증강(Augmentation) 전략 수립**

프로젝트 초기, 전처리 단계에서 시각적 변수를 추가하여 학습 데이터의 다양성을 높이는 전략을 수립함.

 - **추가 코드 및 로직:** <br>

    `v2.RandomRotation(degrees=(0, 360)) # 알약이 어떤 방향으로 놓여도 잘 인식하게 됨`

    `v2.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 2.0)) # 화질이 안 좋은 실전 사진에서도 잘 작동하게 함`

**성과**: 초기 확신 부족으로 PR은 보류하였으나, 이후 YOLO 모델 성능 향상의 자신감의 요인이 됨.

**Phase 2: Faster R-CNN & Optuna를 통한 오류 탐색**

자동 최적화 도구인 Optuna를 도입하여 모델의 한계를 테스트 함.

- **실험 내용**: Faster R-CNN 모델 기반 Optuna 하이퍼파라미터 튜닝 시도.

- **발견 사항**: 배치 사이즈(Batch Size)를 상향 조정하는 과정에서 특정 데이터의 손상 및 오류를 발견.

In [ ]:
!pip install optuna

import os
import optuna
import torch
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# 1. 실험 설계도 (objective 함수)
def objective(trial):
    # 하이퍼파라미터 탐색 범위 설정
    lr = trial.suggest_float("lr", 1e-5, 5e-4, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    optimizer_name = trial.suggest_categorical("optimizer", ["AdamW", "SGD"])
    batch_size = trial.suggest_categorical("batch_size", [4, 8]) 

    # 데이터 로더 (배치 사이즈 변경 반영)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    
    # 모델 정의
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    model.to(DEVICE)

    # 옵티마이저 설정
    params = [p for p in model.parameters() if p.requires_grad]
    if optimizer_name == "AdamW":
        optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    else:
        optimizer = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)

    # --- [학습 루프: 3에폭만 수행] ---
    model.train() # 학습 모드 유지
    for epoch in range(3): 
        for images, targets in train_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            optimizer.zero_grad()
            losses.backward()
            optimizer.step()

    # --- [검증 루프: 에러 방지를 위해 eval()을 하지 않음] ---
    val_loss = 0
    # model.eval()을 하면 에러가 나므로 생략합니다.
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            
            # 훈련 모드 상태여야 loss_dict.values()를 가져올 수 있습니다.
            loss_dict = model(images, targets) 
            val_loss += sum(loss for loss in loss_dict.values()).item()

    # 메모리 정리 (GPU 자원 확보)
    del model
    torch.cuda.empty_cache()
    
    return val_loss / len(val_loader)

# 2. 실제 실험 시작
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10) # 10번의 시도를 진행합니다.

# 3. 결과 출력
print("-" * 30)
print("★ 최적의 파라미터 찾기 완료! ★")
print(study.best_params)
print("-" * 30)

In [ ]:
# 1. 최적의 파라미터 적용
BEST_LR = 3.145051347361507e-05
BEST_WD = 0.001207256183259905
BEST_BATCH_SIZE = 4

# 2. 데이터 로더 재설정 (배치 사이즈 4 적용)
train_loader = DataLoader(train_dataset, batch_size=BEST_BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BEST_BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# 3. 모델 정의
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
model.to(DEVICE)

# 4. 옵티마이저 설정 (AdamW + 최적의 수치)
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=BEST_LR, weight_decay=BEST_WD)

# 5. 최종 학습 (15에폭 추천)
num_epochs = 15 
print(f"최종 승부 시작! (배치 사이즈: {BEST_BATCH_SIZE}, 에폭: {num_epochs})")

for epoch in range(num_epochs):
    model.train()
    train_loss_sum = 0.0
    
    for images, targets in train_loader:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        train_loss_sum += losses.item()

    avg_loss = train_loss_sum / len(train_loader)
    print(f"[Epoch {epoch+1}/{num_epochs}] Average Loss: {avg_loss:.4f}")

    # 매 에폭마다 저장 (나중에 점수 안 나오면 이전 에폭 모델로 돌려막기용)
    torch.save(model.state_dict(), f"final_optuna_model_epoch_{epoch}.pth")

In [ ]:
# 1. 모델 가중치 로드 (가장 마지막 에폭인 14번 사용)
model.load_state_dict(torch.load("final_optuna_model_epoch_14.pth"))
model.to(DEVICE)
model.eval()

# 2. 테스트 파일 목록 확인
test_files = sorted([f for f in os.listdir(TEST_IMG_DIR) if f.endswith(".png")])
rows = []
annotation_id = 1
score_threshold = 0.3 # 베이스라인 기준

print(f"최종 추론 시작 (총 {len(test_files)}장)...")

with torch.no_grad():
    for f in tqdm(test_files):
        img_path = os.path.join(TEST_IMG_DIR, f)
        image = Image.open(img_path).convert("RGB")
        image_id = os.path.splitext(f)[0]
        
        img_tensor = T.ToTensor()(image).to(DEVICE)
        outputs = model([img_tensor])[0]

        keep = outputs["scores"].cpu() >= score_threshold
        boxes  = outputs["boxes"].cpu()[keep]
        labels = outputs["labels"].cpu()[keep]
        scores = outputs["scores"].cpu()[keep]

        for box, lab, sc in zip(boxes, labels, scores):
            x1, y1, x2, y2 = box.tolist()
            
            # [핵심] 강사님 가이드: 모델 번호 -> 원본 번호 변환 후 +1
            orig_cat = model2orig[int(lab.item())]

            rows.append({
                "annotation_id": annotation_id,
                "image_id": image_id,
                "category_id": int(orig_cat + 1), ########## +1 적용! ##########
                "bbox_x": x1,
                "bbox_y": y1,
                "bbox_w": x2 - x1,
                "bbox_h": y2 - y1,
                "score": float(sc.item()),
            })
            annotation_id += 1

# 3. 데이터프레임 정리 및 저장
df_sub = pd.DataFrame(rows)
# 베이스라인처럼 이미지당 상위 4개만 남기기 (선택사항이나 권장함)
df_sub = df_sub.sort_values(by=["image_id", "score"], ascending=[True, False])
df_sub = df_sub.groupby("image_id").head(4)

# 드라이브 경로에 저장
output_path = os.path.join(extract_path, "submission_OPTUNA_AUG_FINAL.csv")
df_sub.to_csv(output_path, index=False)

print(f"✅ 제출 파일 생성 완료: {output_path}")

**Phase 3: 베이스라인 피벗 및 라벨링 오류 디버깅**

팀의 전략이 Baseline 중심으로 변경되었으며 augmentation과 sampler를 추가한 코드를 기반으로 YOLOv11 실험에 착수함.

- **문제 해결**: 초기 YOLOv11 실험 시 Kaggle 0점이 발생하는 문제 직면.

- **원인 분석**: 단순 CSV 변환 오류가 아닌, 학습 단계의 라벨링 코드(Label Encoding) 문제를 발견하여 수정.

- **결과**: 0점 문제를 해결.

In [ ]:
def convert_to_yolo_labels(df, save_dir):
    print(f" 라벨 변환 중: {save_dir}")
    for img_path, group in tqdm(df.groupby("image_path")):
        # 1. 이미지 크기 확인 (정규화 필수)
        with Image.open(img_path) as img:
            w_img, h_img = img.size
        
        file_name = os.path.basename(img_path)
        label_name = os.path.splitext(file_name)[0] + ".txt"
        label_path = os.path.join(save_dir, label_name)

**Phase 4: YOLO 시리즈 비교 실험 (Comparative Study)**

팀 내 최고점(0.99451, YOLOv8s) 모델을 기준으로 동일 조건 하에 모델 교체 실험을 수행.

- **비교 모델**: YOLOv11n, YOLOv8m(epoch 50)
   - YOLOv11n: 0.97910
   - YOLOv8m: 0.98833

- **실험 결과**: YOLOv8s 모델이 현재 데이터셋의 복잡도와 연산 효율성 측면에서 가장 최적의 밸런스를 가짐을 입증.

**Phase 5: 초정밀 하이퍼파라미터 튜닝 (Final Push)**

YOLOv8s 모델의 성능을 0.001점이라도 더 올리기 위한 미세 조정 작업을 수행.

**주요 튜닝 파라미터 및 실험 근거**:

- 기하학적 강건성 확보 (Augmentation Strategy)

   - degrees, translate: 알약의 임의 회전 및 위치 변화에 대응하기 위한 시각적 변수 조정.

   - mixup, overlap_mask: 알약이 겹쳐 있거나 밀집된 환경에서의 탐지 정확도 개선.

- 학습 안정화 및 최적화 (Optimization)

   - Cos_lr: 학습률의 부드러운 감쇠를 통해 최적 수렴 지점(Global Minimum) 안착 유도.

   - Optimizer (AdamW): 가중치 업데이트의 효율성을 높여 정교한 모델 학습 도모.

   - Label Smoothing: 모델의 과적합을 방지하고 일반화(Generalization) 성능 향상.

- 최종 정밀도 마감 (Fine-tuning)

   - Close_mosaic: 학습 종료 전 증강 기법을 제어하여 원본 데이터에 대한 최종 정밀 복습 수행.

   - TTA (Test Time Augmentation): 추론 단계에서 다각도 검토를 적용하여 결과값의 신뢰도 확보.



**실험 결과 요약**

- **결론**: 이러한 일련의 미세 조정 과정을 시도해봤지만 최고 점수인 0.99451에서 더 이상의 개선은 보지 못 함.

가운데 수치만 조절함

In [ ]:
config = {
    "run_name"       : "exp08-yolov8s-finetune", # 실험 번호 업데이트
    "model"          : "yolov8s",
    "num_epochs"     : 20,
    "batch_size"     : 16,
    "lr"             : 1e-4, 
    "weight_decay"   : 5e-4,         # 1e-4에서 살짝 올려 과적합 방지
    
    "cos_lr"         : True,         # 학습률을 코사인 곡선으로 부드럽게 깎아 정밀도 향상
    "label_smoothing": 0.05,         # 모델이 너무 100% 확신하지 않게 하여 일반화 성능 UP
    "close_mosaic"   : 5,            # 마지막 5에폭은 모자이크를 끄고 '정밀 복습'
    "optimizer"      : "AdamW",      # 더 똑똑한 최적화 알고리즘 (지원될 경우)
    
    "lr_step_size"   : 3,            # 만약 cos_lr을 안 쓴다면 이 수치를 유지
    "lr_gamma"       : 0.1,
    "score_threshold": 0.3,
    "augmentation"   : "HFlip+VFlip+ColorJitter+Grayscale+Normalize",
    "num_classes"    : num_classes,
}

In [ ]:
config = {
    "run_name"       : "exp09-yolov8s-finetune", # 실험 번호 업데이트
    "model"          : "yolov8s",
    "num_epochs"     : 20,
    "batch_size"     : 16,
    "lr"             : 1e-4, 
    "weight_decay"   : 5e-4,         # 1e-4에서 살짝 올려 과적합 방지
    
    "copy_paste"     : 0.3,          # 알약 붙여넣기 증강 (강력 추천)
    "box"            : 10.0,         # 박스 위치 정확도 가중치 상향
    "label_smoothing": 0.1,          # 일반화 성능 향상
    "cos_lr"         : True,         # 학습률 부드러운 감소
    "overlap_mask"   : True,         # 겹친 객체 처리 최적화
    "close_mosaic"   : 5,            # 마지막 5에폭은 모자이크를 끄고 '정밀 복습' 모드
    "optimizer"      : "AdamW",      # 더 똑똑한 최적화 알고리즘 (지원될 경우)
    
    "lr_step_size"   : 3,            # 만약 cos_lr을 안 쓴다면 이 수치를 유지
    "lr_gamma"       : 0.1,
    "score_threshold": 0.3,
    "augmentation"   : "HFlip+VFlip+ColorJitter+Grayscale+Normalize",
    "num_classes"    : num_classes,
}

In [ ]:
config = {
    "run_name"       : "exp10-yolov8s-finetune", # 실험 번호 업데이트
    "model"          : "yolov8s",
    "num_epochs"     : 20,
    "batch_size"     : 16,
    "lr"             : 1e-4, 
    "weight_decay"   : 5e-4,         # 1e-4에서 살짝 올려 과적합 방지
    
    "copy_paste"     : 0.0,          # 짧은 학습엔 독이 될 수 있어 제외
    "box"            : 7.5,         # 기본값으로 복원 (안정성)
    "label_smoothing": 0.05,          # 0.1보다는 낮게, 0보다는 높게 (밸런스)
    "cos_lr"         : True,         # 학습률 부드러운 감소
    "overlap_mask"   : True,         # 겹친 객체 처리 최적화
    "close_mosaic"   : 5,            # 마지막 5에폭은 모자이크를 끄고 '정밀 복습' 모드
    "optimizer"      : "AdamW",      # 더 똑똑한 최적화 알고리즘 (지원될 경우)
    
    "lr_step_size"   : 3,            # 만약 cos_lr을 안 쓴다면 이 수치를 유지
    "lr_gamma"       : 0.1,
    "score_threshold": 0.3,
    "augmentation"   : "HFlip+VFlip+ColorJitter+Grayscale+Normalize",
    "num_classes"    : num_classes,
}

In [ ]:
config = {
    "run_name"       : "exp11-yolov8s-finetune", # 실험 번호 업데이트
    "model"          : "yolov8s",
    "num_epochs"     : 20,
    "batch_size"     : 16,
    "lr"             : 1e-4, 
    "weight_decay"   : 5e-4,         # 1e-4에서 살짝 올려 과적합 방지
    
    "degrees"        : 90.0,         # 알약을 0~90도 사이로 마음껏 돌려가며 학습
    "translate"      : 0.1,          # 사진을 살짝씩 밀어서 학습 (위치 정확도 향상)
    "mixup"          : 0.1,          # 사진 두 장을 겹쳐서 학습
    "copy_paste"     : 0.0,          # 짧은 학습엔 독이 될 수 있어 제외
    "box"            : 7.5,          # 기본값으로 복원 (안정성)
    "label_smoothing": 0.05,         # 0.1보다는 낮게, 0보다는 높게 (밸런스)
    "cos_lr"         : True,         # 학습률 부드러운 감소
    "overlap_mask"   : True,         # 겹친 객체 처리 최적화
    "close_mosaic"   : 5,            # 마지막 5에폭은 모자이크를 끄고 '정밀 복습' 모드
    "optimizer"      : "AdamW",      # 더 똑똑한 최적화 알고리즘 (지원될 경우)
    
    "lr_step_size"   : 3,            # 만약 cos_lr을 안 쓴다면 이 수치를 유지
    "lr_gamma"       : 0.1,
    "score_threshold": 0.3,
    "augmentation"   : "HFlip+VFlip+ColorJitter+Grayscale+Normalize",
    "num_classes"    : num_classes,
}